# Predictive Maintenance of Industrial Machines Using Machine Learning

This notebook explains the reasoning before every major implementation step.

## 1. Engineering problem

Machine sensors and controllers generate measurements. We want to learn patterns that separate normal operation (`0`) from a failure condition (`1`). This is binary classification.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from predictive_maintenance import *

df = load_and_clean_data(ROOT / 'data' / 'ai4i2020.csv')

## 2. Inspect before modelling

We first examine size, columns, sample rows, data types, missing values, duplicates and the target balance. This prevents us from blindly fitting a model.

In [ ]:
print('Shape:', df.shape)
display(df.head())
display(df.dtypes.to_frame('data type'))
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
display(df[TARGET].value_counts().rename(index={0: 'Normal', 1: 'Failure'}).to_frame('count'))
display(df[NUMERIC_FEATURES].describe().T)

## 3. Select valid inputs

`UDI` and `Product ID` are identifiers, not condition measurements. `TWF`, `HDF`, `PWF`, `OSF` and `RNF` directly determine the target, so using them would leak the answer. We keep only product type and five operating measurements.

In [ ]:
X = df[FEATURES]
y = df[TARGET]
print('Inputs:', FEATURES)
print('Target:', TARGET)

## 4. Exploratory data analysis

The charts show imbalance and relationships between physical measurements. A chart suggests patterns, but does not prove causation.

In [ ]:
create_eda_charts(df)
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
df[TARGET].value_counts().sort_index().plot.bar(ax=axes[0], title='Class distribution')
df.plot.scatter('Rotational speed [rpm]', 'Torque [Nm]', c=TARGET, cmap='coolwarm', alpha=.25, ax=axes[1], title='Torque vs speed')
df.boxplot(column='Tool wear [min]', by=TARGET, ax=axes[2])
axes[2].set_title('Tool wear by condition')
plt.suptitle('')
plt.tight_layout()

## 5. Train and test

The helper uses an 80/20 stratified split, so both sets retain the rare failure class. The tree is deliberately shallow and uses balanced class weights.

In [ ]:
pipeline, metrics = evaluate_model(df)
metrics

## 6. Interpret the confusion matrix

Sensitivity = TP/(TP+FN): how many actual failures we caught. Specificity = TN/(TN+FP): how many healthy cases we correctly left alone. Missing a failure can damage equipment or affect safety; too many false alarms waste maintenance time and production capacity.

In [ ]:
cm = [[metrics['true_negatives'], metrics['false_positives']], [metrics['false_negatives'], metrics['true_positives']]]
print('Confusion matrix:', cm)
print(f"Sensitivity: {metrics['sensitivity_recall']:.3f}")
print(f"Specificity: {metrics['specificity']:.3f}")
print(f"Precision: {metrics['precision']:.3f}")
print(f"F1-score: {metrics['f1_score']:.3f}")

## 7. Demonstrate one prediction

In a real system, these values could arrive from sensors and a controller. Here we enter one illustrative set of readings.

In [ ]:
reading = {'Type': 'L', 'Air temperature [K]': 300.0, 'Process temperature [K]': 310.0, 'Rotational speed [rpm]': 1350, 'Torque [Nm]': 60.0, 'Tool wear [min]': 210}
predict_machine_condition(pipeline, reading)

## 8. Conclusion and limitation

This proof of concept shows how Data Science turns measurements from a mechatronic system into a maintenance warning. Because the dataset is synthetic, the model must not be treated as a production safety system. Real deployment needs real sensor history, engineering validation and continuous monitoring.